In [0]:
%run ./00_config

In [0]:
# 2. Define the explicit source and destination file paths
source_staging_path = f"{volume_root_path}/staging/transactions_batch_02.csv"
target_ingest_path  = f"{path_transactions}/transactions_batch_02.csv"

print("🚚 Promoting transactions_batch_02.csv to the active ingestion folder...")

try:
    # dbutils.fs.mv cuts the file from source and pastes it into the destination
    # This automatically removes it from staging/ in a single transaction block
    file_moved = dbutils.fs.mv(source_staging_path, target_ingest_path)
    
    if file_moved:
        print(f"  🟢 SUCCESS: File safely moved to: {target_ingest_path}")
    else:
        print("  ⚠️ WARNING: File movement returned False. Check if file was already moved.")

except Exception as e:
    print(f"  ❌ ERROR: File promotion failed: {str(e)}")

# 3. Mandated Verification Check: Prove staging/ is now completely empty
print("\n🔍 Verification: Auditing staging directory contents...")
staging_contents = dbutils.fs.ls(f"{volume_root_path}/staging/")

if len(staging_contents) == 0:
    print("  🟢 SUCCESS: The staging directory is completely empty.")
else:
    print("  ❌ ALERT: Staging is NOT empty! Hidden files still exist:")
    for item in staging_contents:
        print(f"    └─ Found leftover file: {item.path} ({item.size} bytes)")

# 4. Ingestion Folder Verification Check: Prove batch_02 is visible to transactions
print("\n📋 Current active Ingestion Folder Manifest:")
transactions_contents = dbutils.fs.ls(path_transactions)
for file_info in transactions_contents:
    print(f"  🔹 File: {file_info.name:<30} | Size: {file_info.size / 1024:>7.2f} KB")


In [0]:
df = spark.table("bluepeak.bronze.transactions_raw")
grouped = df.groupBy("_source_file").count()
display(grouped)

In [0]:
%sql
SELECT * FROM bluepeak.ops.ingestion_audit;

In [0]:
# ============================================================================
# US-1.12: Live Transaction Key Overlap & Row Variance Audit
# ============================================================================
print("🔍 Auditing physical row collisions inside bronze.transactions_raw...")

# Find keys that appear in both source files and compare their data strings
overlap_check_df = spark.sql(f"""
  WITH batch_1 AS (
    SELECT * FROM {table_bronze_transactions} 
    WHERE _source_file LIKE '%transactions_batch_01.csv'
  ),
  batch_2 AS (
    SELECT * FROM {table_bronze_transactions} 
    WHERE _source_file LIKE '%transactions_batch_02.csv'
  )
  SELECT 
    b1.transaction_id,
    b1.amount AS batch_1_amount,
    b2.amount AS batch_2_amount,
    CASE 
      WHEN b1.account_id = b2.account_id 
       AND b1.txn_timestamp = b2.txn_timestamp 
       AND b1.amount = b2.amount 
       THEN 'Identical Row Data'
      ELSE 'Mismatched (Differing Data Fields)'
    END AS row_variance_status
  FROM batch_1 b1
  INNER JOIN batch_2 b2 ON b1.transaction_id = b2.transaction_id
""")

print(f"📊 Total Overlapping Transaction Keys Found: {overlap_check_df.count()}")
display(overlap_check_df)


In [0]:
%sql
-- Run this check query to verify your final clean-room metrics
SELECT 'Customers' AS dataset, COUNT(*) AS row_count FROM bluepeak.bronze.customers_raw
UNION ALL
SELECT 'Accounts', COUNT(*) FROM bluepeak.bronze.accounts_raw
UNION ALL
SELECT 'Branches', COUNT(*) FROM bluepeak.bronze.branches_raw
UNION ALL
SELECT 'Transactions (Combined Batches 1 & 2)', COUNT(*) FROM bluepeak.bronze.transactions_raw;


In [0]:
reconciliation_query = f"""
WITH target_counts AS (
  SELECT _source_file, COUNT(*) AS bronze_rows FROM {table_bronze_customers} GROUP BY _source_file
  UNION ALL
  SELECT _source_file, COUNT(*) FROM {table_bronze_accounts} GROUP BY _source_file
  UNION ALL
  SELECT _source_file, COUNT(*) FROM {table_bronze_branches} GROUP BY _source_file
  UNION ALL
  SELECT _source_file, COUNT(*) FROM {table_bronze_transactions} GROUP BY _source_file
),
audit_logs AS (
  SELECT 
    CASE 
      WHEN source_path LIKE '%customers' THEN concat(source_path, '/customers.csv')
      WHEN source_path LIKE '%accounts' THEN concat(source_path, '/accounts.csv')
      WHEN source_path LIKE '%branches' THEN concat(source_path, '/branches.csv')
      WHEN source_path LIKE '%transactions' AND rows_loaded = 2330 THEN concat(source_path, '/transactions_batch_01.csv')
      WHEN source_path LIKE '%transactions' AND rows_loaded = 1099 THEN concat(source_path, '/transactions_batch_02.csv')
      ELSE source_path
    END AS simulated_file_path,
    rows_loaded AS file_rows
  FROM {table_audit_reconcile}
)
SELECT 
  element_at(split(t._source_file, '/'), -1) AS file_name,
  a.file_rows AS rows_in_file,
  t.bronze_rows AS rows_in_bronze,
  CASE 
    WHEN a.file_rows = t.bronze_rows THEN 'PASS'
    ELSE 'FAIL'
  END AS reconciliation_result
FROM target_counts t
INNER JOIN audit_logs a 
  ON t._source_file LIKE concat('%', element_at(split(a.simulated_file_path, '/'), -1))
ORDER BY file_name ASC
"""

reconciliation_df = spark.sql(reconciliation_query)
display(reconciliation_df)